# 伊那中学校マインクエスト
## システム開発と少しAIの話

左の ▶ ボタンを押すと、その場所のプログラムが動きます。


---
## 1. まずは1回動かしてみる

下の ▶ を押してみて。`""` の中の文字は、好きに書き換えてOK。


In [2]:
print("伊那中のみなさん、こんにちは！")


伊那中のみなさん、こんにちは！


---
## 2. 高速バスの予約、あれどうなってる？

スマホで伊那 ⇄ 新宿のバスを予約するとき、画面には「残り3席」とか出る。

- **満席かどうか、どうやって分かってる？**
- **同じ席が2人に売れちゃったら、どうなる？**


---
# 3. システムの構成

システムは、だいたい3つの部品でできている。

`フロントエンド（画面）` → `バックエンド（サーバー）` → `データベース（記録）`

ボタンを押すと、右へ「お願い」が流れ、左へ「答え」が返ってくる。
奥（データベース）から順に見ていく。

#### ① データベース ── 覚えているところ

「9/5 8時の便の 1A席 は、○○さんが予約ずみ」。
こういう事実をぜんぶ書き留めておく係。電源を切っても消えない。

- 覚えるのが仕事。**判断はしない**
- ただし「同じ席を2人に売る」みたいな、あり得ない形は**受け付けない**
- 話しかける言葉が **SQL**

ここが消えると会社は営業できない。だから一番大事。今日つくるのはここ。

#### ② バックエンド ── ルールを判断するところ

画面から「1Aを取りたい」と来たら、決める係。

- 空いているか確かめる
- お金が払われたか確かめる
- ダメなら断る、OKならデータベースに書かせる

画面には出てこない。でも**会社のルールは全部ここに入っている**。

#### ③ フロントエンド ── 人が見て、押すところ

座席表を並べて、押されたらバックエンドにお願いを送る係。

- 見やすく出すのが仕事
- **自分では何も決めない**
- 「残り3席」の 3 も、自分で数えた数字ではなく、奥から届いた数字

> ボタンの向こう側で、この3つが毎回リレーしている。
> 今日つくるのは一番奥、**データベース**。ここが本体です。


---
# 4. データベース実習

「覚えているところ」を自分の手で作る。ここが本体。

### 4-1. 準備：データベースを起動する

1〜2分かかります。押したら待っててください。


In [8]:
import subprocess

def _ok(args):
    """その接続方法でMySQLに繋がるか試す。"""
    try:
        return subprocess.run(["mysql", *args, "-u", "root", "-e", "SELECT 1"],
                              capture_output=True).returncode == 0
    except FileNotFoundError:
        return False

# ソケット（Colab）か TCP（手元の docker compose）か、繋がるほうを使う
MYSQL = next((a for a in ([], ["-h", "127.0.0.1"]) if _ok(a)), None)

if MYSQL is None:   # MySQLがまだ無い = Colab。ここで入れる（1〜2分）
    !apt-get -qq update > /dev/null && apt-get -qq install -y mysql-server > /dev/null
    !service mysql start
    MYSQL = []

subprocess.run(["mysql", *MYSQL, "-u", "root", "-e", "CREATE DATABASE IF NOT EXISTS bus"])

def sql(q):
    """SQLを実行して結果を表示する。以降ぜんぶこれを使う。"""
    r = subprocess.run(["mysql", *MYSQL, "-u", "root", "--table", "bus"],
                       input=q, capture_output=True, text=True)
    print(r.stdout or r.stderr)

print("準備OK", "（ソケット接続）" if not MYSQL else "（127.0.0.1 接続）")


準備OK （127.0.0.1 接続）


### 4-2. テーブルを作る（CREATE TABLE）

データベースは「表」でできている。この表のことを **テーブル** と呼ぶ。
バス予約に必要な表は、たった4つ。

| テーブル | なにが入っているか |
|---|---|
| `car` | バスの車両。「伊那号 1号車」 |
| `car_chairs` | その車両のイス。1A, 1B ... 毎日変わらない |
| `car_plan` | 運行の予定。9/5 8:00 伊那→新宿 |
| `car_plan_chairs` | **便ごとのイス。ここに予約が入る** |

**なぜイスの表が2つあるの？**
イスは毎日同じ場所にある。でも「誰が座るか」は便ごとに違う。
だから分ける。これを考えるのが *設計* という仕事。


In [9]:
sql("""
DROP TABLE IF EXISTS car_plan_chairs, car_plan, car_chairs, car;

-- 1) バスの車両
CREATE TABLE car (
  id   INT AUTO_INCREMENT PRIMARY KEY,
  name VARCHAR(50) NOT NULL COMMENT '車両の呼び名'
);

-- 2) その車両のイス（動かない情報）
CREATE TABLE car_chairs (
  id          INT AUTO_INCREMENT PRIMARY KEY,
  car_id      INT         NOT NULL,
  seat_no     VARCHAR(5)  NOT NULL COMMENT '1A, 1B ...',
  window_side BOOLEAN     NOT NULL COMMENT '窓側なら true',
  UNIQUE (car_id, seat_no),
  FOREIGN KEY (car_id) REFERENCES car(id)
);

-- 3) 運行の予定（いつ・どこからどこへ・どの車両で）
CREATE TABLE car_plan (
  id           INT AUTO_INCREMENT PRIMARY KEY,
  car_id       INT         NOT NULL,
  departure_at DATETIME    NOT NULL,
  origin       VARCHAR(50) NOT NULL,
  destination  VARCHAR(50) NOT NULL,
  price        INT         NOT NULL,
  FOREIGN KEY (car_id) REFERENCES car(id)
);

-- 4) 便ごとのイス = 予約が入る場所
CREATE TABLE car_plan_chairs (
  id             INT AUTO_INCREMENT PRIMARY KEY,
  car_plan_id    INT         NOT NULL,
  car_chair_id   INT         NOT NULL,
  passenger_name VARCHAR(50) DEFAULT NULL COMMENT 'NULL なら空席',
  reserved_at    DATETIME    DEFAULT NULL,
  UNIQUE (car_plan_id, car_chair_id),  -- 同じ便の同じ席は1行だけ
  FOREIGN KEY (car_plan_id)  REFERENCES car_plan(id),
  FOREIGN KEY (car_chair_id) REFERENCES car_chairs(id)
);

SHOW TABLES;
""")


+-----------------+
| Tables_in_bus   |
+-----------------+
| car             |
| car_chairs      |
| car_plan        |
| car_plan_chairs |
+-----------------+



### 4-3. データを入れる（INSERT）

表ができた。まだ空っぽなので、中身を入れる。

注目してほしいのは最後のほう。
**8席 × 2便 = 16行** を手で書かずに、SQL に作らせている。
「人間がやると間違えるところは、機械にやらせる」——これも仕事のコツ。


In [10]:
sql("""
INSERT INTO car (name) VALUES ('伊那号 1号車'), ('伊那号 2号車');

INSERT INTO car_chairs (car_id, seat_no, window_side) VALUES
 (1,'1A',true),(1,'1B',false),(1,'1C',false),(1,'1D',true),
 (1,'2A',true),(1,'2B',false),(1,'2C',false),(1,'2D',true);

INSERT INTO car_plan (car_id, departure_at, origin, destination, price) VALUES
 (1,'2026-09-05 08:00:00','伊那','新宿',3800),
 (1,'2026-09-05 14:00:00','新宿','伊那',3800);

-- 便 × イス を手打ちしない（16行をSQLに作らせる）
INSERT INTO car_plan_chairs (car_plan_id, car_chair_id)
SELECT p.id, c.id FROM car_plan p JOIN car_chairs c ON c.car_id = p.car_id;

-- 2人ぶんの予約が入った状態にする
UPDATE car_plan_chairs SET passenger_name='小川', reserved_at=NOW()
 WHERE car_plan_id=1 AND car_chair_id=(SELECT id FROM car_chairs WHERE car_id=1 AND seat_no='1A');
UPDATE car_plan_chairs SET passenger_name='田中', reserved_at=NOW()
 WHERE car_plan_id=1 AND car_chair_id=(SELECT id FROM car_chairs WHERE car_id=1 AND seat_no='2D');

SELECT COUNT(*) AS `入った座席の数` FROM car_plan_chairs;
""")


+-----------------------+
| 入った座席の数        |
+-----------------------+
|                    16 |
+-----------------------+



### 4-4. 調べる（SELECT）

ここからが本番。`SELECT` は「知りたいことを聞く」命令。

**どんな便がある？**


In [11]:
sql("""
SELECT p.id, c.name, p.departure_at, p.origin, p.destination, p.price
  FROM car_plan p
  JOIN car c ON c.id = p.car_id
 ORDER BY p.departure_at;
""")


+----+-------------------+---------------------+--------+-------------+-------+
| id | name              | departure_at        | origin | destination | price |
+----+-------------------+---------------------+--------+-------------+-------+
|  1 | 伊那号 1号車      | 2026-09-05 08:00:00 | 伊那   | 新宿        |  3800 |
|  2 | 伊那号 1号車      | 2026-09-05 14:00:00 | 新宿   | 伊那        |  3800 |
+----+-------------------+---------------------+--------+-------------+-------+



### 4-5. 8時の便の座席表（予約サイトのあの画面）


In [12]:
sql("""
SELECT ch.seat_no,
       IF(ch.window_side,'窓側','通路側') AS `席`,
       IFNULL(pc.passenger_name,'空席')  AS `状況`
  FROM car_plan_chairs pc
  JOIN car_chairs ch ON ch.id = pc.car_chair_id
 WHERE pc.car_plan_id = 1
 ORDER BY ch.seat_no;
""")


+---------+-----------+--------+
| seat_no | 席        | 状況   |
+---------+-----------+--------+
| 1A      | 窓側      | 小川   |
| 1B      | 通路側    | 空席   |
| 1C      | 通路側    | 空席   |
| 1D      | 窓側      | 空席   |
| 2A      | 窓側      | 空席   |
| 2B      | 通路側    | 空席   |
| 2C      | 通路側    | 空席   |
| 2D      | 窓側      | 田中   |
+---------+-----------+--------+



### 4-6. 「残り○席」はこう数えている


In [13]:
sql("""
SELECT p.departure_at, p.origin, p.destination,
       COUNT(*)                          AS `全席`,
       SUM(pc.passenger_name IS NULL)     AS `空席`
  FROM car_plan p
  JOIN car_plan_chairs pc ON pc.car_plan_id = p.id
 GROUP BY p.id
 ORDER BY p.departure_at;
""")


+---------------------+--------+-------------+--------+--------+
| departure_at        | origin | destination | 全席   | 空席   |
+---------------------+--------+-------------+--------+--------+
| 2026-09-05 08:00:00 | 伊那   | 新宿        |      8 |      6 |
| 2026-09-05 14:00:00 | 新宿   | 伊那        |      8 |      8 |
+---------------------+--------+-------------+--------+--------+



### 4-7. 「窓側の空いてる席だけ見せて」

人間が全部の席を目で見て探す代わりに、1行聞けば出てくる。
これが何万席あっても同じ速さで出る。だからシステムを作る意味がある。


In [14]:
sql("""
SELECT p.departure_at, ch.seat_no
  FROM car_plan_chairs pc
  JOIN car_chairs ch ON ch.id = pc.car_chair_id
  JOIN car_plan   p  ON p.id  = pc.car_plan_id
 WHERE pc.passenger_name IS NULL
   AND ch.window_side = true
 ORDER BY p.departure_at, ch.seat_no;
""")


+---------------------+---------+
| departure_at        | seat_no |
+---------------------+---------+
| 2026-09-05 08:00:00 | 1D      |
| 2026-09-05 08:00:00 | 2A      |
| 2026-09-05 14:00:00 | 1A      |
| 2026-09-05 14:00:00 | 1D      |
| 2026-09-05 14:00:00 | 2A      |
| 2026-09-05 14:00:00 | 2D      |
+---------------------+---------+



### 4-8. わざとエラーを出してみる

最初の質問に戻る。**同じ席が2人に売れたら？**

答え: 売れない。データベースが拒否する。やってみよう。


In [15]:
# 8時の便の 1A席 を、もう一度作ろうとする
sql("""
INSERT INTO car_plan_chairs (car_plan_id, car_chair_id) VALUES (1, 1);
""")

# => ERROR 1062 Duplicate entry ... と出るのが「正しい」動き


ERROR 1062 (23000) at line 2: Duplicate entry '1-1' for key 'car_plan_chairs.car_plan_id'



さっき書いた `UNIQUE (car_plan_id, car_chair_id)` の1行が効いている。

> **人間が気をつける** のではなく、**間違えられない形にしておく**。

これがエンジニアの仕事のいちばん面白いところ。
エラーは失敗じゃなくて、システムがちゃんと守ってくれた合図。


### 4-9. やってみよう

自分の名前で、好きな席を予約してみる。`'あなたの名前'` と `'1C'` を書き換えて実行。
そのあと 4-5 と 4-6 のセルをもう一度実行すると、空席が減っているはず。


In [ ]:
sql("""
UPDATE car_plan_chairs SET passenger_name='あなたの名前', reserved_at=NOW()
 WHERE car_plan_id=1
   AND car_chair_id=(SELECT id FROM car_chairs WHERE car_id=1 AND seat_no='1C');

SELECT ch.seat_no, IFNULL(pc.passenger_name,'空席') AS `状況`
  FROM car_plan_chairs pc JOIN car_chairs ch ON ch.id = pc.car_chair_id
 WHERE pc.car_plan_id = 1 ORDER BY ch.seat_no;
""")


---
# 5. バックエンド実習

データベースは、言われたことをやるだけ。**「やっていいか」を決めるのはバックエンド**。

「1Bを取りたい」と来たときに、
1. そんな席ある？
2. もう誰か座ってない？
3. OKならデータベースに書く / ダメなら断る

これを Python で書いてみる。まずは結果を Python で受け取る道具から。


In [ ]:
def q(query):
    """SQLを実行して、結果を Python のリスト（表）で受け取る。"""
    r = subprocess.run(["mysql", *MYSQL, "-u", "root", "-N", "-B", "bus"],
                       input=query, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stderr.strip())
    return [line.split("\t") for line in r.stdout.splitlines()]

q("SELECT seat_no, window_side FROM car_chairs WHERE car_id = 1")


### 5-1. 予約のルールを書く

さっきの1〜3を、そのまま関数にする。**これがバックエンドの正体**。


In [ ]:
def reserve(plan_id, seat_no, name):
    """予約していいか判断して、OKならデータベースに書く。"""
    name = name.replace("'", "")   # 本物の現場では「プレースホルダ」を使う。今日は簡単に

    found = q(f"""
      SELECT pc.id, pc.passenger_name
        FROM car_plan_chairs pc
        JOIN car_chairs ch ON ch.id = pc.car_chair_id
       WHERE pc.car_plan_id = {plan_id} AND ch.seat_no = '{seat_no}'
    """)

    if not found:                                   # 1. そんな席ある？
        return f"✕ {seat_no} という席はありません"

    row_id, who = found[0]
    if who != "NULL":                               # 2. もう誰か座ってない？
        return f"✕ {seat_no} は {who} さんが予約ずみです"

    q(f"UPDATE car_plan_chairs SET passenger_name='{name}', reserved_at=NOW() WHERE id={row_id}")
    return f"○ {seat_no} を {name} さんで予約しました"   # 3. OK


### 5-2. 動かして、わざと断らせる

3回呼んでみる。2回目と3回目は**断られるのが正しい**。


In [ ]:
print(reserve(1, "1B", "きみの名前"))
print(reserve(1, "1B", "あとから来た人"))   # 同じ席をもう一度 → 断られる
print(reserve(1, "9Z", "だれか"))          # 無い席 → 断られる


4章で見た「データベースが拒否する」との違いに注目。

| | 断り方 |
|---|---|
| データベース | エラーで止まる（最後の砦） |
| バックエンド | **理由を言って、やさしく断る**（人に見せる用） |

両方いる。バックエンドが忘れても、データベースが最後に止めてくれる。

### 5-3. やってみよう

ルールを1つ増やしてみる。下のセルの `# ここに追加` の行を書き換えて、
もう一度 5-1 のセルを実行し直すと、新しいルールが効く。

- 例1: 名前が空っぽなら断る
- 例2: 「予約ずみ」ではなく「キャンセル待ちに入れますか？」と返す


In [ ]:
# 例1をやってみた版（コピーして 5-1 の reserve に入れてもいい）
def check_name(name):
    if not name.strip():
        return "✕ 名前を入れてください"
    if len(name) > 20:
        return "✕ 名前が長すぎます"
    # ここに追加
    return None

print(check_name(""), check_name("小川"))


---
# 6. フロントエンド実習

最後は、人が見るところ。フロントエンドの仕事は2つだけ。

1. 奥からもらったデータを、**見やすく並べる**
2. 押されたら、バックエンドに**お願いを送る**

自分では何も決めない。まず「並べる」だけやってみる。


In [1]:
from IPython.display import HTML, display

def seat_map(plan_id=1):
    """座席表を絵にする。判断はしない、ただ並べるだけ。"""
    rows = q(f"""
      SELECT ch.seat_no, IFNULL(pc.passenger_name, '')
        FROM car_plan_chairs pc
        JOIN car_chairs ch ON ch.id = pc.car_chair_id
       WHERE pc.car_plan_id = {plan_id}
       ORDER BY ch.seat_no
    """)
    boxes = "".join(
        f'<div style="padding:12px;border-radius:6px;text-align:center;'
        f'background:{"#f8bbd0" if who else "#c8e6c9"}">{seat}<br><small>{who or "空席"}</small></div>'
        for seat, who in rows)
    return HTML(f'<div style="display:grid;grid-template-columns:repeat(4,90px);gap:6px">{boxes}</div>')

display(seat_map())


NameError: name 'q' is not defined

緑が空席、ピンクが予約ずみ。
**この色は画面が勝手に決めているのではなく、データベースの中身そのもの。**

### 6-1. 押せるようにする

ボタンを押すと、5章の `reserve()` が呼ばれ、その中でデータベースが書き換わる。
`フロントエンド → バックエンド → データベース` が、1回のクリックで全部つながる。


In [ ]:
import ipywidgets as widgets

name_box = widgets.Text(value="あなたの名前", description="なまえ")
log = widgets.Output()

def make_button(seat_no, who):
    b = widgets.Button(description=f"{seat_no} {who or '空席'}",
                       button_style="danger" if who else "success",
                       disabled=bool(who))
    def on_click(_):
        msg = reserve(1, seat_no, name_box.value)   # ← 画面は「お願い」するだけ
        with log:
            log.clear_output()
            print(msg)
        if msg.startswith("○"):                     # OKだった時だけ見た目を変える
            b.description = f"{seat_no} {name_box.value}"
            b.button_style, b.disabled = "danger", True
    b.on_click(on_click)
    return b

seats = q("""
  SELECT ch.seat_no, IFNULL(pc.passenger_name, '')
    FROM car_plan_chairs pc JOIN car_chairs ch ON ch.id = pc.car_chair_id
   WHERE pc.car_plan_id = 1 ORDER BY ch.seat_no
""")
display(name_box,
        widgets.GridBox([make_button(s, w) for s, w in seats],
                        layout=widgets.Layout(grid_template_columns="repeat(4, 130px)")),
        log)


---
# 7. AI

## 7-1. ChatからAgentへ

少し前のAIは **チャット** だった。質問すると、答えが返ってくる。それだけ。
書くのも、動かすのも、直すのも人間。

今のAIは **エージェント** になった。
「バスの予約システムを作って」と頼むと、AIが自分でファイルを開き、
コードを書き、実際に動かし、エラーが出たら直す。**最後までやりきる。**

| | チャット | エージェント |
|---|---|---|
| できること | 答えを返す | 自分で手を動かす |
| ファイルを開く | 人間 | AI |
| 動かして確かめる | 人間 | AI |
| 何を作るか決める | 人間 | **人間** |

今日みんなが1つずつ書いたSQL、エージェントに任せれば数十秒で全部書ける。


## 7-2. システム開発とCoding Agent

**Coding Agent** = コードを書くことに特化したエージェント。
今の現場では、エンジニアはこれと2人で組んで仕事をしている。

- 昔: エンジニアが1行ずつ手で打つ
- 今: **AIが下書きし、人が決めて、人が確かめる**

これは未来の話じゃなくて、今の現場の話。（実はこの教材も、AIと一緒に作った）

決まった形のコードを書く部分は、もうAIの担当。
今日やった「イスの表を2つに分ける」みたいな **設計の判断** は、まだ人間の担当。


## 7-3. AIは仕事を奪うのか

歴史に似た話がある。**明治維新**。
それまで一番有利な立場にいた武士が、時代が変わった瞬間に窮地に立たされた。

今回は、**コンピューターの中だけで完結する仕事** をしていた人が同じ場所に立っている。
これは誰かが悪いのではなく、**抗えない自然な流れ**。

ただ、こうも思う。
この数十年、コンピューターだけで高い収入を得られたこと自体が、少し不思議なことだった。
だから今起きているのは、**世の中の揺り戻し** だと考えている。


## 7-4. 仕事と、みんなの時代

**そもそも仕事って何だろう。**

親に育ててもらい、社会に出て、まずは自分を養えるようになる。
次に部下を持ち、家族を持ち、今度は **自分が育てる立場** になる。
仕事は、その順番を進んでいくためのもの。

先生の世代は「AIを使い始めた世代」。
みんなは **最初からAIが隣にいる世代** になる。

だから、AIに勝とうとしなくていい。
**どんな分野の仕事でも、AIと組んでプラスアルファを生み出す世界。**
とても楽しい未来が待っているはず。

道具は変わる。**作りたいものを決める力**は変わらない。


---
# 8. 今日のまとめ

- システムは3つ。**データベース**（覚える）→ **バックエンド**（決める）→ **フロントエンド**（見せる）
- バス予約の裏側は、4つの表と、それに聞く命令（SQL）でできている
- 「間違えられない形にしておく」のがエンジニアの仕事
- AIは強力な道具。使う側になればいい

**このノートブックは家でも無料で使えます。**
ファイル > ドライブにコピーを保存 → 自分のものになる。好きにいじってOK。

質問どうぞ。
